In [1]:
import numpy as np
import numpy.typing as npt
from astropy.time import Time
from sorts.propagator import SGP4
from sorts.space_object import SpaceObject
from sorts.radar.radars import get_radar
from sorts.types import Datetime64_us, Timedelta64_us, Float64_as_sec
from sorts.controller_v2.tracker_controller import TrackerController
from sorts.controller_v2.fence_scan_controller_new import FenceScanController
from sorts.schedule_v2 import Schedule, ExperimentDetail
from sorts.scheduler_v2.priority_scheduling import _priority_scheduling_df

# import for plottings
import pandas as pd
from sorts.plotting_deps import lp, lp_geo_data, geodatasets, gpd
from sorts import plots

The geodata is provided by © OpenStreetMap contributors and is made available here under the Open Database License (ODbL).


In [2]:
pd.set_option("display.expand_frame_repr", False)

In [3]:
epoch = Time(53005.0, format="mjd", scale="utc")  # 2004-01-01 00:00:00Z
# start_time = Time("2025-06-30 00:00:00")
# end_time = Time("2025-06-30 00:00:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms
start_time = Time("2025-01-01 04:04:00")
end_time = Time("2025-01-01 04:04:01")
control_slice_duration = np.timedelta64(10_000, "us")  # 10ms
# start_time = Time("2025-01-01 02:45:00")
# end_time = Time("2025-01-01 06:15:00")
# control_slice_duration = np.timedelta64(int(60 * 1e6), "us")

eiscat3d = get_radar("eiscat3d", "stage1-array")

spobj = SpaceObject(
    SGP4,
    propagator_options={"settings": {"out_frame": "ITRF"}},
    a=7200e3,
    e=0.02,
    i=75,
    raan=86,
    aop=0,
    mu0=60,
    epoch=epoch,
    parameters={"d": 0.1},
)


exp_detail_0 = ExperimentDetail(
    id=0,
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,
    noise_temp=150.0,
    slice_duration=control_slice_duration
)

exp_detail_1 = ExperimentDetail(
    id=1,
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,
    noise_temp=150.0,
    slice_duration=control_slice_duration
)

time_arr: npt.NDArray[Datetime64_us] = np.arange(
    start_time.to_value("datetime64").astype("datetime64[us]"),  # type: ignore
    end_time.to_value("datetime64").astype("datetime64[us]"),  # type: ignore
    control_slice_duration,
)
time_arr = time_arr[::4] # TODO: remove; strided to bring up the effects of scheduling
dt_arr: npt.NDArray[Timedelta64_us] = time_arr - epoch.to_value("datetime64").astype("datetime64[us]")  # type: ignore
dsec_arr: npt.NDArray[Float64_as_sec] = dt_arr.astype(np.float64) / 1e6  # type: ignore

ecefs = spobj.get_state(dsec_arr)

trackerController = TrackerController(
    tx_station=eiscat3d.tx[0],
    rx_stations=[],
    time=time_arr,
    space_object_states=ecefs,
    exp_detail=exp_detail_0,
    # azimuth_range=None,
    # elevation_range=None,
)

fenceScanController = FenceScanController(
    tx_station=eiscat3d.tx[0],
    rx_station=[],
    exp_datail=exp_detail_1,
    azimuth=90, # sweep from east to west
    min_elevation=30,
    pointings_per_cycle=40,
)

In [4]:
plots.ecef_states_positions_plot(ecefs)

In [5]:
tracker_schs = trackerController.generate()
tracker_tx_sch_df = tracker_schs.tx_schedule.as_dataframe()
tracker_tx_sch_df

,stt_tstmp_us,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 04:04:00.000,176.908697,83.511849,0,2025-01-01 04:04:00.010
1,2025-01-01 04:04:00.040,176.896913,83.513734,0,2025-01-01 04:04:00.050
2,2025-01-01 04:04:00.080,176.885122,83.515619,0,2025-01-01 04:04:00.090
3,2025-01-01 04:04:00.120,176.873323,83.517504,0,2025-01-01 04:04:00.130
4,2025-01-01 04:04:00.160,176.861518,83.519388,0,2025-01-01 04:04:00.170
5,2025-01-01 04:04:00.200,176.849706,83.521272,0,2025-01-01 04:04:00.210
6,2025-01-01 04:04:00.240,176.837887,83.523156,0,2025-01-01 04:04:00.250
7,2025-01-01 04:04:00.280,176.826062,83.525040,0,2025-01-01 04:04:00.290
8,2025-01-01 04:04:00.320,176.814229,83.526923,0,2025-01-01 04:04:00.330
9,2025-01-01 04:04:00.360,176.802389,83.528806,0,2025-01-01 04:04:00.370


In [6]:
fence_schs = fenceScanController.generate(start_time, end_time)
fence_tx_sch_df = fence_schs.tx_schedule.as_dataframe()
fence_tx_sch_df

,stt_tstmp_us,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 04:04:00.000,90.0,30.000000,1,2025-01-01 04:04:00.010
1,2025-01-01 04:04:00.010,90.0,33.076923,1,2025-01-01 04:04:00.020
2,2025-01-01 04:04:00.020,90.0,36.153846,1,2025-01-01 04:04:00.030
3,2025-01-01 04:04:00.030,90.0,39.230769,1,2025-01-01 04:04:00.040
4,2025-01-01 04:04:00.040,90.0,42.307692,1,2025-01-01 04:04:00.050
...,...,...,...,...,...
95,2025-01-01 04:04:00.950,90.0,76.153846,1,2025-01-01 04:04:00.960
96,2025-01-01 04:04:00.960,90.0,79.230769,1,2025-01-01 04:04:00.970
97,2025-01-01 04:04:00.970,90.0,82.307692,1,2025-01-01 04:04:00.980
98,2025-01-01 04:04:00.980,90.0,85.384615,1,2025-01-01 04:04:00.990


In [7]:
# check schedule df memory usage
fence_tx_sch_df.memory_usage().sum()/1e6

np.float64(0.004128)

In [8]:
tx_sch = tracker_schs.tx_schedule
plots.azel_polar_plot(tx_sch.pointing_az, tx_sch.pointing_el)

In [9]:
tx_sch = fence_schs.tx_schedule
plots.azel_polar_plot(tx_sch.pointing_az, tx_sch.pointing_el)

In [10]:
(tracker_schs.tx_schedule.meta, fence_schs.tx_schedule.meta)

({0: ExperimentDetail(id=0, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(10000,'us'))},
 {1: ExperimentDetail(id=1, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(10000,'us'))})

In [11]:
master_sch_meta = {0: exp_detail_0, 1: exp_detail_1}
master_sch_df = _priority_scheduling_df([tracker_schs.tx_schedule, fence_schs.tx_schedule])
master_sch_df

,stt_tstmp_us,pointing_az,pointing_el,exp_num,end_time,allowed_start_time,allowed_end_time,is_overlaped
0,2025-01-01 04:04:00.000,176.908697,83.511849,0,2025-01-01 04:04:00.010,2025-01-01 04:04:00.000,2025-01-01 04:04:00.020,False
2,2025-01-01 04:04:00.020,90.000000,36.153846,1,2025-01-01 04:04:00.030,2025-01-01 04:04:00.010,2025-01-01 04:04:00.040,False
1,2025-01-01 04:04:00.040,176.896913,83.513734,0,2025-01-01 04:04:00.050,2025-01-01 04:04:00.030,2025-01-01 04:04:00.060,False
6,2025-01-01 04:04:00.060,90.000000,48.461538,1,2025-01-01 04:04:00.070,2025-01-01 04:04:00.050,2025-01-01 04:04:00.080,False
2,2025-01-01 04:04:00.080,176.885122,83.515619,0,2025-01-01 04:04:00.090,2025-01-01 04:04:00.070,2025-01-01 04:04:00.100,False
10,2025-01-01 04:04:00.100,90.000000,60.769231,1,2025-01-01 04:04:00.110,2025-01-01 04:04:00.090,2025-01-01 04:04:00.120,False
3,2025-01-01 04:04:00.120,176.873323,83.517504,0,2025-01-01 04:04:00.130,2025-01-01 04:04:00.110,2025-01-01 04:04:00.140,False
14,2025-01-01 04:04:00.140,90.000000,73.076923,1,2025-01-01 04:04:00.150,2025-01-01 04:04:00.130,2025-01-01 04:04:00.160,False
4,2025-01-01 04:04:00.160,176.861518,83.519388,0,2025-01-01 04:04:00.170,2025-01-01 04:04:00.150,2025-01-01 04:04:00.180,False
18,2025-01-01 04:04:00.180,90.000000,85.384615,1,2025-01-01 04:04:00.190,2025-01-01 04:04:00.170,2025-01-01 04:04:00.200,False


In [12]:
master_sch = Schedule.from_dataframe(master_sch_df, master_sch_meta)
master_sch

Schedule(meta={0: ExperimentDetail(id=0, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(10000,'us')), 1: ExperimentDetail(id=1, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(10000,'us'))}, stt_tstmp_us=array(['2025-01-01T04:04:00.000000', '2025-01-01T04:04:00.020000',
       '2025-01-01T04:04:00.040000', '2025-01-01T04:04:00.060000',
       '2025-01-01T04:04:00.080000', '2025-01-01T04:04:00.100000',
       '2025-01-01T04:04:00.120000', '2025-01-01T04:04:00.140000',
       '2025-01-01T04:04:00.160000', '2025-01-01T04:04:00.180000',
       '2025-01-01T04:04:00.200000', '2025-01-01T04:04:00.220000',
       '2025-01-01T04:04:00.240000', '2025-01-01T04:04:00.260000',
       '2025-01-01T04:04:00.280000', '2025-01-01T04:04:00.300000',
       '2025-01-01T04:04:00.320000'

In [16]:
plots.schedule_plot(master_sch)